# Uzbek (uzb) — Full NLP Pipeline with Custom Stanza Models

Uzbek uses the Latin script since 1995 (official). The Cyrillic variant remains in wide use. TurkicNLP provides Apertium FST morphology (Stable), custom-trained Stanza neural models for POS tagging, lemmatisation, and dependency parsing, bidirectional Cyrillic↔Latin transliteration, and NLLB-200 embeddings/translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('uzb')

## 2. Cyrillic ↔ Latin Transliteration (1995 official standard)

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

cyrl = "Мен мактабга бораман."
print("Detected:", detect_script(cyrl))

t = Transliterator("uzb", source=Script.CYRILLIC, target=Script.LATIN)
latin = t.transliterate(cyrl)
print("Latin:", latin)

t_back = Transliterator("uzb", source=Script.LATIN, target=Script.CYRILLIC)
print("Back:", t_back.transliterate(latin))

## 3. Morphological Analysis (Apertium FST — Stable)

In [ ]:
nlp = Pipeline(
    "uzb",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)

# Latin input
doc = nlp("Men maktabga boraman.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

In [ ]:
# Cyrillic input with auto-detection
nlp_cyrl = Pipeline(
    "uzb",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_cyrl("Мен мактабга бораман.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (Custom Stanza)

TurkicNLP includes custom-trained Stanza models for Uzbek, providing POS tagging, lemmatisation, and dependency parsing.

In [ ]:
nlp_parse = Pipeline(
    "uzb",
    processors=["tokenize", "pos", "lemma", "depparse"],
)

doc = nlp_parse("Men maktabga ketdim.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Head':<5} {'Deprel'}")
print("-" * 60)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.head!s:<5} {w.deprel}")

## 5. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "uzb",
    processors=["tokenize", "morph", "pos", "lemma", "depparse"],
    morph_backend="apertium",
)
doc = nlp_full("O'zbekiston Markaziy Osiyodagi eng yirik davlatlardan biri.")
print(doc.to_conllu())

## 6. Translation via NLLB-200

In [ ]:
turkicnlp.download("uzb", processors=["translate"])
trans = Pipeline("uzb", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("O'zbekiston Markaziy Osiyodagi davlat.")
print("EN:", doc.translation)